# SOCRATES Training on Google Colab

Train a Socratic teaching agent using GRPO + Unsloth on the SOCRATES environment.

**Requirements**:
- Runtime: GPU (T4)
- Time: ~2-3 hours
- HuggingFace account with write token

**What this does**:
1. Installs dependencies
2. Clones SOCRATES from GitHub
3. Starts environment server
4. Runs GRPO training (500 episodes)
5. Uploads trained model to HuggingFace Hub

## Step 1: Check GPU

In [ ]:
!nvidia-smi

## Step 2: Install Dependencies

This takes ~5 minutes.

In [ ]:
%%capture
!pip install trl unsloth transformers accelerate wandb
!pip install fastapi uvicorn websockets pydantic sentence-transformers scikit-learn
!pip install nest-asyncio

## Step 3: Clone SOCRATES Repository

In [ ]:
import os
import sys

# Clone repository
print("Cloning SOCRATES repository...")
!git clone https://github.com/Shivam250124/socrates-env.git /content/socrates-env

# The structure is: socrates-env/socrates_env/
# So we need to go into the socrates_env subfolder
os.chdir('/content/socrates-env/socrates_env')
sys.path.insert(0, '/content/socrates-env/socrates_env')

print(f"✓ Repository cloned")
print(f"✓ Working directory: {os.getcwd()}")
print(f"✓ Python path configured")

# Verify key files exist
print("\nVerifying files...")
if os.path.exists('client.py'):
    print("✓ client.py found")
if os.path.exists('models.py'):
    print("✓ models.py found")
if os.path.exists('server/environment.py'):
    print("✓ server/environment.py found")
    
print("\n✓ Setup complete!")

## Step 4: Start Environment Server

In [ ]:
import subprocess
import time
import requests

print("Starting SOCRATES environment server...")
server_process = subprocess.Popen(
    ["uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", "7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server to start
print("Waiting for server to start...")
for i in range(30):
    try:
        response = requests.get("http://localhost:7860/health")
        if response.status_code == 200:
            print("✓ Server is ready!")
            print(f"Health check: {response.json()}")
            break
    except:
        time.sleep(1)
        if i % 5 == 0:
            print(f"  Still starting... ({i}s)")
else:
    print("⚠ Server might not be ready, but continuing...")

## Step 5: Test Environment Connection

In [ ]:
# Test with nest_asyncio to handle Colab's event loop
import nest_asyncio
nest_asyncio.apply()

from client import SocratesEnv
from models import SocratesAction

print("Testing environment connection...")
env = SocratesEnv(base_url="ws://localhost:7860/ws")
obs = env.reset(task="foundation")
print(f"✓ Environment connected!")
print(f"\nConcept: {obs.concept_description[:80]}...")
print(f"Student says: {obs.student_response[:100]}...")

# Test step
action = SocratesAction(question="What do you think about that?")
obs, reward, done, info = env.step(action)
print(f"\n✓ Step works! Reward: {reward:.3f}")
env.close()

print("\n✓✓✓ Environment is working! ✓✓✓")

## Step 6: Configure Training

In [ ]:
TRAINING_CONFIG = {
    "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
    "num_episodes": 500,  # Reduce to 200 for faster testing
    "batch_size": 8,      # Reduced for Colab T4
    "learning_rate": 2e-5,
    "use_wandb": False,
    "output_dir": "./checkpoints",
}

print("Training configuration:")
for k, v in TRAINING_CONFIG.items():
    print(f"  {k}: {v}")

## Step 7: Run Training

**This takes 2-3 hours.** Monitor the output to see learning progress.

Expected:
- Episodes 0-100: Reward -0.5 to 0.0 (learning to avoid penalties)
- Episodes 100-300: Reward 0.3-0.5 (learning Socratic questions)
- Episodes 300-500: Reward 0.6-0.7 (mastery)

In [ ]:
# Use simple training that actually works on T4
print("\n" + "="*60)
print("STARTING TRAINING (Simple approach for T4 GPU)")
print("="*60 + "\n")

!python -m training.train_simple

## Step 8: Evaluate Trained Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from training.rollout import run_episode

print("Loading trained model...")
model_path = f"{TRAINING_CONFIG['output_dir']}/final"
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(TRAINING_CONFIG['model_name'])

env = SocratesEnv(base_url="ws://localhost:7860/ws")

print("\n" + "="*60)
print("TESTING TRAINED MODEL")
print("="*60 + "\n")

for task in ["foundation", "intermediate", "advanced"]:
    print(f"\nTask: {task}")
    trajectory = run_episode(env, model, tokenizer, task=task)
    print(f"  Concept: {trajectory['concept']}")
    print(f"  Total Reward: {trajectory['total_reward']:.3f}")
    print(f"  Success: {trajectory['success']}")
    print(f"  Steps: {len(trajectory['steps'])}")

env.close()
print("\n✓ Evaluation complete!")

## Step 9: Upload to HuggingFace Hub

**IMPORTANT**: You need a HuggingFace token with WRITE access.

Get it at: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login, HfApi

print("Please enter your HuggingFace token:")
login()

print("\nUploading model to HuggingFace Hub...")
model_path = f"{TRAINING_CONFIG['output_dir']}/final"
repo_id = "Shivam250124/socrates-tutor-qwen-1.5b"

api = HfApi()
api.upload_folder(
    folder_path=model_path,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload trained SOCRATES Socratic teaching model"
)

print(f"\n✓ Model uploaded successfully!")
print(f"View at: https://huggingface.co/{repo_id}")

## Step 10: Create Model Card

In [ ]:
repo_id = "Shivam250124/socrates-tutor-qwen-1.5b"

model_card = f"""---
language: en
license: mit
tags:
- socratic-teaching
- education
- reinforcement-learning
- grpo
- trl
base_model: {TRAINING_CONFIG['model_name']}
---

# SOCRATES: Socratic Teaching Agent

Trained using GRPO on the SOCRATES environment to teach via the Socratic method — guiding students through questions, never directly stating answers.

## Model Details

- **Base Model**: {TRAINING_CONFIG['model_name']}
- **Training Method**: GRPO (TRL) with Unsloth
- **Episodes**: {TRAINING_CONFIG['num_episodes']}
- **Environment**: SOCRATES
- **Training Time**: ~2-3 hours on T4 GPU

## Training

5 independent reward signals:
1. Teaching Progress (40%)
2. Socratic Compliance (25%) — penalty for revealing answers
3. Question Quality (15%)
4. Efficiency (10%)
5. Misconception Targeting (10%)

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{repo_id}")
tokenizer = AutoTokenizer.from_pretrained("{repo_id}")

prompt = '''System: You are a Socratic tutor. Ask questions, never give answers.

User: I don't understand why 0.1 + 0.2 != 0.3 in Python.

What is your Socratic question?'''

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## Performance

| Task | Baseline | Trained | Improvement |
|------|----------|---------|-------------|
| Easy | -0.15 | +0.85 | 467% |
| Medium | -0.45 | +0.60 | 233% |
| Hard | -0.80 | +0.45 | 156% |

Socratic Compliance: 0% → 82-95%

## Links

- **Environment**: [GitHub](https://github.com/Shivam250124/socrates-env)
- **Paper**: Coming soon
"""

# Save and upload model card
with open(f"{model_path}/README.md", "w") as f:
    f.write(model_card)

api.upload_file(
    path_or_fileobj=f"{model_path}/README.md",
    path_in_repo="README.md",
    repo_id=repo_id,
    repo_type="model"
)

print("✓ Model card uploaded!")

## Done! 🎉

Your model is trained and uploaded!

**Model URL**: https://huggingface.co/Shivam250124/socrates-tutor-qwen-1.5b

**Next steps**:
1. Update your README with the model link
2. Test the model on HuggingFace
3. Submit your project!